In [1]:
!pip install -q decord

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 60.0 MB/s eta 0:00:00


In [2]:
# Install once if needed:
# !pip install -q transformers accelerate decord pillow tqdm

import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from transformers import (
    BlipProcessor,
    BlipForImageTextRetrieval
)

from decord import VideoReader, cpu
from google.colab import drive


# =========================================================
# GOOGLE DRIVE
# =========================================================

drive.mount("/content/drive")


# =========================================================
# SETTINGS
# =========================================================

DATA_ROOT = (
    "/content/drive/Shared drives/"
    "DATA 298A/DATA/MSVD"
)

MODEL_NAME = "Salesforce/blip-itm-base-coco"

NUM_FRAMES = 8
VIDEO_BATCH_SIZE = 8
TEXT_BATCH_SIZE = 64
MAX_TEXT_LEN = 40

# Average the strongest two frames for each video.
TOP_K_FRAMES = 2

OUTPUT_JSON = "/content/blip_eval_results.json"


# =========================================================
# HELPERS
# =========================================================

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def sample_frame_indices(
    total_frames: int,
    num_frames: int
) -> List[int]:

    if total_frames <= 0:
        raise ValueError("Video has no frames.")

    actual_num_frames = min(
        num_frames,
        total_frames
    )

    return np.linspace(
        0,
        total_frames - 1,
        actual_num_frames,
        dtype=int
    ).tolist()


def load_video_frames(
    video_path: str,
    num_frames: int
) -> List[Image.Image]:

    video_reader = VideoReader(
        video_path,
        ctx=cpu(0)
    )

    indices = sample_frame_indices(
        len(video_reader),
        num_frames
    )

    frames = video_reader.get_batch(
        indices
    ).asnumpy()

    return [
        Image.fromarray(frame).convert("RGB")
        for frame in frames
    ]


# =========================================================
# BUILD TEST SET
# ONE QUERY PER VIDEO
# =========================================================

def build_test_set(
    data_root: str
) -> Tuple[List[Dict], List[Dict]]:

    test_file = os.path.join(
        data_root,
        "msvd_test.json"
    )

    video_root = os.path.join(
        data_root,
        "raw_videos"
    )

    if not os.path.exists(test_file):
        raise FileNotFoundError(
            f"Missing file: {test_file}"
        )

    if not os.path.exists(video_root):
        raise FileNotFoundError(
            f"Missing folder: {video_root}"
        )

    records = load_json(test_file)

    videos = {}
    first_caption_by_video = {}

    for item in records:

        video_name = item["video"]

        video_id = str(
            item.get(
                "video_id",
                video_name
            )
        )

        video_path = os.path.join(
            video_root,
            video_name
        )

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path
            }

        if video_id not in first_caption_by_video:

            captions = item["caption"]

            if isinstance(captions, str):
                captions = [captions]

            if len(captions) > 0:
                caption = " ".join(
                    str(captions[0])
                    .strip()
                    .split()
                )

                if caption:
                    first_caption_by_video[
                        video_id
                    ] = caption

    queries = []

    for video_id, caption in (
        first_caption_by_video.items()
    ):
        queries.append({
            "video_id": video_id,
            "caption": caption
        })

    return queries, list(videos.values())


# =========================================================
# BLIP MODEL
# =========================================================

class BLIPVideoTextModel:

    def __init__(
        self,
        model_name: str,
        device: torch.device
    ):
        self.device = device

        self.processor = (
            BlipProcessor.from_pretrained(
                model_name
            )
        )

        self.model = (
            BlipForImageTextRetrieval
            .from_pretrained(
                model_name
            )
            .to(device)
        )

        self.model.eval()

    @torch.inference_mode()
    def encode_images(
        self,
        images: List[Image.Image]
    ) -> torch.Tensor:

        inputs = self.processor(
            images=images,
            return_tensors="pt"
        )

        pixel_values = inputs[
            "pixel_values"
        ].to(self.device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=self.device.type == "cuda"
        ):
            vision_output = (
                self.model.vision_model(
                    pixel_values=pixel_values,
                    return_dict=True
                )
            )

            image_features = (
                vision_output
                .last_hidden_state[:, 0, :]
            )

            image_embeddings = (
                self.model.vision_proj(
                    image_features
                )
            )

            image_embeddings = F.normalize(
                image_embeddings,
                dim=-1
            )

        return image_embeddings

    @torch.inference_mode()
    def encode_texts(
        self,
        texts: List[str]
    ) -> torch.Tensor:

        inputs = self.processor(
            text=texts,
            padding=True,
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )

        input_ids = inputs[
            "input_ids"
        ].to(self.device)

        attention_mask = inputs[
            "attention_mask"
        ].to(self.device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=self.device.type == "cuda"
        ):
            text_output = (
                self.model.text_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_dict=True
                )
            )

            text_features = (
                text_output
                .last_hidden_state[:, 0, :]
            )

            text_embeddings = (
                self.model.text_proj(
                    text_features
                )
            )

            text_embeddings = F.normalize(
                text_embeddings,
                dim=-1
            )

        return text_embeddings


# =========================================================
# ENCODE VIDEOS
# =========================================================

@torch.inference_mode()
def encode_videos(
    blip,
    videos,
    num_frames,
    batch_size
):

    all_video_embeddings = []
    video_ids = []

    for start in tqdm(
        range(0, len(videos), batch_size),
        desc="Encoding videos"
    ):

        batch_videos = videos[
            start:start + batch_size
        ]

        batch_frames = []
        frame_counts = []
        valid_batch_videos = []

        for video in batch_videos:

            try:
                frames = load_video_frames(
                    video["video_path"],
                    num_frames
                )
            except Exception as error:
                print(
                    "Skipping:",
                    video["video_name"],
                    error
                )
                continue

            if len(frames) == 0:
                continue

            batch_frames.extend(frames)
            frame_counts.append(len(frames))
            valid_batch_videos.append(video)

        if len(batch_frames) == 0:
            continue

        frame_embeddings = blip.encode_images(
            batch_frames
        )

        frame_start = 0

        for video, frame_count in zip(
            valid_batch_videos,
            frame_counts
        ):

            frame_end = (
                frame_start + frame_count
            )

            video_frame_embeddings = (
                frame_embeddings[
                    frame_start:frame_end
                ]
            )

            # Store all frame embeddings.
            # Query-dependent top-k pooling happens later.
            all_video_embeddings.append(
                video_frame_embeddings.cpu()
            )

            video_ids.append(
                video["video_id"]
            )

            frame_start = frame_end

    return all_video_embeddings, video_ids


# =========================================================
# ENCODE QUERIES
# =========================================================

@torch.inference_mode()
def encode_queries(
    blip,
    queries,
    batch_size
):

    all_embeddings = []
    ground_truth_video_ids = []

    for start in tqdm(
        range(0, len(queries), batch_size),
        desc="Encoding texts"
    ):

        batch = queries[
            start:start + batch_size
        ]

        texts = [
            query["caption"]
            for query in batch
        ]

        embeddings = blip.encode_texts(
            texts
        )

        all_embeddings.append(
            embeddings.cpu()
        )

        ground_truth_video_ids.extend([
            query["video_id"]
            for query in batch
        ])

    return (
        torch.cat(all_embeddings, dim=0),
        ground_truth_video_ids
    )


# =========================================================
# SIMILARITY
# =========================================================

def compute_similarity(
    text_embeddings,
    video_frame_embeddings,
    top_k_frames
):

    number_of_queries = (
        text_embeddings.shape[0]
    )

    number_of_videos = len(
        video_frame_embeddings
    )

    similarity = np.zeros(
        (
            number_of_queries,
            number_of_videos
        ),
        dtype=np.float32
    )

    for video_index, frames in enumerate(
        tqdm(
            video_frame_embeddings,
            desc="Computing similarities"
        )
    ):

        frame_scores = (
            text_embeddings
            @ frames.T
        )

        k = min(
            top_k_frames,
            frame_scores.shape[1]
        )

        strongest_scores = torch.topk(
            frame_scores,
            k=k,
            dim=1
        ).values

        video_scores = strongest_scores.mean(
            dim=1
        )

        similarity[
            :,
            video_index
        ] = video_scores.numpy()

    return similarity


# =========================================================
# METRICS
# =========================================================

def compute_metrics(
    similarity,
    ground_truth_video_ids,
    video_ids
):

    video_id_to_index = {
        video_id: index
        for index, video_id
        in enumerate(video_ids)
    }

    valid_query_indices = []
    ground_truth_indices = []

    for query_index, video_id in enumerate(
        ground_truth_video_ids
    ):

        if video_id in video_id_to_index:

            valid_query_indices.append(
                query_index
            )

            ground_truth_indices.append(
                video_id_to_index[video_id]
            )

    similarity = similarity[
        valid_query_indices
    ]

    ground_truth_indices = np.asarray(
        ground_truth_indices,
        dtype=np.int64
    )

    sorted_indices = np.argsort(
        -similarity,
        axis=1
    )

    ranks = []

    for query_index, correct_index in enumerate(
        ground_truth_indices
    ):

        rank = int(
            np.where(
                sorted_indices[
                    query_index
                ] == correct_index
            )[0][0]
        ) + 1

        ranks.append(rank)

    ranks = np.asarray(ranks)

    return {
        "R@1": float(
            np.mean(ranks <= 1)
        ),
        "R@5": float(
            np.mean(ranks <= 5)
        ),
        "R@10": float(
            np.mean(ranks <= 10)
        ),
        "MRR": float(
            np.mean(1.0 / ranks)
        ),
        "MeanRank": float(
            np.mean(ranks)
        ),
        "MedianRank": float(
            np.median(ranks)
        ),
        "top1_accuracy": float(np.mean(ranks <= 1)),
        "top5_accuracy": float(np.mean(ranks <= 5))
    }


# =========================================================
# MAIN
# =========================================================

def main():

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print("Device:", device)

    queries, videos = build_test_set(
        DATA_ROOT
    )

    print(
        f"Queries: {len(queries)} | "
        f"Videos: {len(videos)}"
    )

    blip = BLIPVideoTextModel(
        MODEL_NAME,
        device
    )

    print(
        "Model:",
        blip.model.__class__.__name__
    )

    start_time = time.time()

    video_frame_embeddings, video_ids = (
        encode_videos(
            blip=blip,
            videos=videos,
            num_frames=NUM_FRAMES,
            batch_size=VIDEO_BATCH_SIZE
        )
    )

    text_embeddings, gt_video_ids = (
        encode_queries(
            blip=blip,
            queries=queries,
            batch_size=TEXT_BATCH_SIZE
        )
    )

    similarity = compute_similarity(
        text_embeddings=text_embeddings,
        video_frame_embeddings=(
            video_frame_embeddings
        ),
        top_k_frames=TOP_K_FRAMES
    )

    metrics = compute_metrics(
        similarity=similarity,
        ground_truth_video_ids=(
            gt_video_ids
        ),
        video_ids=video_ids
    )

    elapsed_seconds = (
        time.time() - start_time
    )

    metrics["total_seconds"] = float(
        elapsed_seconds
    )

    print("\nBLIP Retrieval Metrics")

    for name, value in metrics.items():
        print(f"{name}: {value:.4f}")

    print(
        f"\nRuntime: "
        f"{elapsed_seconds / 60:.2f} minutes"
    )

    with open(
        OUTPUT_JSON,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            metrics,
            file,
            indent=2
        )

    print(
        "Saved to:",
        OUTPUT_JSON
    )


main()

Mounted at /content/drive
Device: cuda
Queries: 670 | Videos: 670


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/456 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Model: BlipForImageTextRetrieval


Encoding videos:   0%|          | 0/84 [00:00<?, ?it/s]

Encoding texts:   0%|          | 0/11 [00:00<?, ?it/s]

Computing similarities:   0%|          | 0/670 [00:00<?, ?it/s]


BLIP Retrieval Metrics
R@1: 0.5060
R@5: 0.8030
R@10: 0.8642
MRR: 0.6342
MeanRank: 7.7940
MedianRank: 1.0000
top1_accuracy: 0.5060
top5_accuracy: 0.8030
total_seconds: 844.7443

Runtime: 14.08 minutes
Saved to: /content/blip_eval_results.json


In [3]:
import json
import pandas as pd

RESULT_PATH = "/content/blip_eval_results.json"

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("BLIP Retrieval Results\n")

# Handle both possible JSON formats
if "metrics" in results:
    metrics = results["metrics"]
else:
    metrics = results

df = pd.DataFrame(
    list(metrics.items()),
    columns=["Metric", "Value"]
)

display(df)

BLIP Retrieval Results



,Metric,Value
0,R@1,0.505970
1,R@5,0.802985
2,R@10,0.864179
3,MRR,0.634215
4,MeanRank,7.794030
5,MedianRank,1.000000
6,top1_accuracy,0.505970
7,top5_accuracy,0.802985
8,total_seconds,844.744326
